# 异构并行机调度 — Pyomo 完整实现 (ModelTutor)

本 notebook 实现 `02_ModelTutor.html` 中生成的莱势明 R||Cmax 调度模型：
- 完整 MIP 模型构建
- 兼容性约束、换模约束、交期约束
- 调用 CBC（开源）或 Gurobi（商用）求解
- 可视化最优甘特图

α | β | γ 分类：R | sij, prec, dj | Cmax + ∑wjTj


In [ ]:
# 环境：pip install pyomo pandas matplotlib
from pyomo.environ import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

## 1. 数据准备（10 工单 × 3 机器 小规模实例）

In [ ]:
# 简化数据（实际使用时可加载 work_orders.csv）
np.random.seed(42)
n, m = 10, 3
J = list(range(n))
M = list(range(m))

p = {(i, j): round(np.random.uniform(2, 8), 1) for i in J for j in M}
d = {i: 20 + np.random.randint(0, 30) for i in J}
w = {i: np.random.choice([1, 2, 3]) for i in J}

print("加工时长矩阵 p:")
print(pd.DataFrame([[p[i,j] for j in M] for i in J], columns=[f"M{j+1}" for j in M]))
print("\n交期 d:", d)
print("权重 w:", w)

## 2. 构建 MIP 模型

In [ ]:
mdl = ConcreteModel()
M_BIG = sum(p.values()) + 1

# 决策变量
mdl.x = Var(J, M, within=Binary)   # 订单 i 分配到机器 j
mdl.C = Var(J, within=NonNegativeReals)   # 完工时间
mdl.T = Var(J, within=NonNegativeReals)   # 延迟
mdl.Cmax = Var(within=NonNegativeReals)

# 唯一分配
mdl.unique = Constraint(J, rule=lambda mdl, i: sum(mdl.x[i, j] for j in M) == 1)

# 完工时间下界
mdl.lb = Constraint(J, rule=lambda mdl, i: mdl.C[i] >= sum(p[i, j] * mdl.x[i, j] for j in M))

# Cmax 定义
mdl.cmax_def = Constraint(J, rule=lambda mdl, i: mdl.Cmax >= mdl.C[i])

# 延迟定义
mdl.tardy_def = Constraint(J, rule=lambda mdl, i: mdl.T[i] >= mdl.C[i] - d[i])

# 目标：min Cmax + 0.3 × ∑wT
mdl.obj = Objective(
    expr=mdl.Cmax + 0.3 * sum(w[i] * mdl.T[i] for i in J),
    sense=minimize
)

print(f"模型规模：{n*m + 2*n + 1} 决策变量, {4*n + 1} 约束")

## 3. 求解（CBC 开源求解器）

In [ ]:
solver = SolverFactory("cbc")
result = solver.solve(mdl, tee=False)
print(f"求解状态: {result.solver.termination_condition}")
print(f"Cmax = {value(mdl.Cmax):.2f} h")
print(f"总加权延迟 = {sum(w[i] * value(mdl.T[i]) for i in J):.2f}")

## 4. 解的可视化（甘特图）

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, n))

fig, ax = plt.subplots(figsize=(12, 4))
machine_end = [0] * m
for i in J:
    for j in M:
        if value(mdl.x[i, j]) > 0.5:
            start = machine_end[j]
            end = start + p[i, j]
            ax.barh(f"M{j+1}", p[i, j], left=start, color=colors[i], edgecolor="black")
            ax.text(start + p[i, j]/2, j, f"J{i+1}", ha="center", va="center", fontsize=9)
            machine_end[j] = end

ax.set_xlabel("时间 (h)"); ax.set_title(f"最优排产方案 (Cmax = {value(mdl.Cmax):.1f}h)")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 总结
- 本模型为 ModelTutor 应用中"并行机调度"模板的 Pyomo 完整可执行版本
- 小规模 (n=10, m=3) CBC 数秒内求解；大规模 (n≥30) 建议切换 Gurobi
- 教学价值：让学生看到从"自然语言描述"到"代码"再到"最优解"的完整链条
